In [1]:
# Upgrade pip first
!pip install --upgrade pip

# Core ML libraries
!pip install torch==2.1.0+cu121 torchvision==0.16.1+cu121 torchaudio==2.1.1 --extra-index-url https://download.pytorch.org/whl/cu121

# Hugging Face and tokenizer tools
!pip install transformers==4.34.0 sentencepiece tokenizers safetensors

# Efficient fine-tuning
!pip install peft bitsandbytes accelerate einops

# Dataset and evaluation
!pip install datasets evaluate

# Experiment tracking (optional but recommended)
!pip install wandb


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 30.6 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu121
ERROR: Could not find a version that satisfies the requirement torch==2.1.0+cu121 (from versions: 2.2.0, 2.2.0+cu121, 2.2.1, 2.2.1+cu121, 2.2.2, 2.2.2+cu121, 2.3.0, 2.3.0+cu121, 2.3.1, 2.3.1+cu121, 2.4.0, 2.4.0+cu121, 2.4.1, 2.4.1+cu121, 2.5.0, 2.5.0+cu121, 2.5.1, 2.5.1+cu121, 2.6.0, 2.7.0, 2.7.1, 2.8.0)
ERROR: No matching distribution found for torch==2.1.0+cu121
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 122.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 114.1 MB/s  0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.34.4
    Uninstalling huggingface-hub-0.34.4:
      Successfully uninstalled huggingface

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForSeq2Seq
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, TaskType
import evaluate

##Experiment 01 with higher rank for LLAMA 3.1

In [3]:
from datasets import load_dataset, DatasetDict

# Load the JSONL file
train_ds = load_dataset("json", data_files="/content/finance_pc_train.jsonl", split="train")

# display the first row
print(train_ds[0])

# Check the column names
print(train_ds.column_names)


Generating train split: 0 examples [00:00, ? examples/s]

{'question': "What services does Iron Mountain provide to protect organizations' information and reduce storage costs?", 'context': 'Iron Mountain helps organizations protect their information and reduce storage costs by storing physical records and data backup media, offering information management solutions, and providing data center space.', 'answer': 'Iron Mountain provides services such as storing physical records and data backup media, offering information management solutions, and providing data center space for enterprise-class colocation and hyperscale deployments.'}
['question', 'context', 'answer']


In [4]:
full_ds = load_dataset("json", data_files="/content/finance_pc_train.jsonl", split="train")

# Split 10% for validation and 90% for the training
split_ds = full_ds.train_test_split(test_size=0.1, seed=42)
train_ds = split_ds["train"]
val_ds = split_ds["test"]

print("Train samples:", len(train_ds))
print("Validation samples:", len(val_ds))
print("Columns:", train_ds.column_names)
print("Sample:", train_ds[0])

Train samples: 900
Validation samples: 100
Columns: ['question', 'context', 'answer']
Sample: {'question': 'What is the significance of Note 13 in the context of legal proceedings described in the Annual Report on Form 10-K?', 'context': 'For a description of our significant pending legal proceedings, see Note 13 titled Commitments and Contingencies - Legal Proceedings of the Notes to Consolidated Financial Statements included in Part II, Item 8 of this Annual Report on Form 10-K.', 'answer': "Note 13 is significant because it contains a detailed description of the company's significant pending legal proceedings."}


In [5]:
from transformers import AutoTokenizer
from datasets import load_dataset
# load the model
model_name = "meta-llama/Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load dataset and split
full_ds = load_dataset("json", data_files="/content/finance_pc_train.jsonl", split="train")
split_ds = full_ds.train_test_split(test_size=0.1, seed=42)
train_ds = split_ds["train"]
val_ds = split_ds["test"]

# Tokenization with teacher forcing
def tokenize_qa(example):
    # Full prompt
    prompt_text = f"Question: {example['question']}\nContext: {example['context']}\nAnswer: "
    answer_text = example['answer'] + tokenizer.eos_token  # add EOS

    # Encode prompt
    prompt_ids = tokenizer(prompt_text, truncation=True, max_length=512)["input_ids"]
    # Encode answer
    answer_ids = tokenizer(answer_text, truncation=True, max_length=256)["input_ids"]

    # Input_ids = prompt + answer
    input_ids = prompt_ids + answer_ids
    # Labels = mask prompt tokens with -100, only keep answer tokens for loss
    labels = [-100] * len(prompt_ids) + answer_ids

    return {"input_ids": input_ids, "labels": labels}

tokenized_train = train_ds.map(tokenize_qa, remove_columns=train_ds.column_names)
tokenized_val = val_ds.map(tokenize_qa, remove_columns=val_ds.column_names)


tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

model_name = "meta-llama/Llama-3.1-8B-Instruct"

# --- Define the quantization configuration ---
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# --- Load the base model in 4-bit ---
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True
)

# --- LoRA configuration for full attention + MLP ---
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ]
)

# --- Apply LoRA to the model ---
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # Should show all attention + MLP LoRA params


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196


In [7]:

import wandb
wandb.login
wandb.init(project="llama3.1_custom", name="finetune_lora_2")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: abhi1199 (abhi1199-city-university-of-london) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [8]:

data_collator = DataCollatorForSeq2Seq(tokenizer, return_tensors="pt", padding=True)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # Perplexity will be computed in the Trainer's evaluate() output
    return {}

In [9]:

training_args = TrainingArguments(
    output_dir="./llama3_qlora_finance",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=50,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,
    fp16=True,
    optim="paged_adamw_32bit",
    eval_strategy="steps",
    eval_steps=500,
    logging_dir="./logs",
    report_to="wandb"  # send metrics to W&B
)

In [10]:

from transformers import TrainerCallback, TrainerState, TrainerControl
import math

class PerplexityLoggerCallback(TrainerCallback):
    def on_evaluate(self, args, state: TrainerState, control: TrainerControl, metrics=None, **kwargs):
        if metrics and "eval_loss" in metrics:
            eval_loss = metrics["eval_loss"]
            perplexity = math.exp(eval_loss)
            wandb.log({"eval_perplexity": perplexity, "step": state.global_step})

In [11]:

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[PerplexityLoggerCallback]
)


/tmp/ipython-input-2783209826.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [12]:

trainer.train()


Step,Training Loss,Validation Loss


Step,Training Loss,Validation Loss


TrainOutput(global_step=171, training_loss=0.42836773325825295, metrics={'train_runtime': 1081.4251, 'train_samples_per_second': 2.497, 'train_steps_per_second': 0.158, 'total_flos': 1.1893923889053696e+16, 'train_loss': 0.42836773325825295, 'epoch': 3.0})

In [13]:
model.save_pretrained("./llama3_qlora_finance")
tokenizer.save_pretrained("./llama3_qlora_finance")
print("Finetuning complete! Model saved.")
wandb.finish()

Finetuning complete! Model saved.


train/epoch,▁▄▇█
train/global_step,▁▄▇█
train/grad_norm,▁█▆
train/learning_rate,█▅▁
train/loss,█▄▁
total_flos,1.1893923889053696e+16
train/epoch,3
train/global_step,171
train/grad_norm,1.14779
train/learning_rate,3e-05
train/loss,0.2043


In [14]:

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch

finetuned_model_path = "./llama3_qlora_finance"
base_model_name = "meta-llama/Llama-3.1-8B-Instruct"

# loading the tokeinizer
tokenizer = AutoTokenizer.from_pretrained(finetuned_model_path)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

# quantized model
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# base model of hf
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    offload_folder="offload"
)

# base model + fine tuned model for testing
model = PeftModel.from_pretrained(base_model, finetuned_model_path)
model.eval()

print("Model loaded successfully!")


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Model loaded successfully!


In [15]:

def ask_model(context, question, max_new_tokens=200, temperature=0.7):
    prompt = (
        f"Context:\n{context}\n\n"
        f"Question:\n{question}\n\n"
        "Answer:"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    answer = tokenizer.decode(output[0], skip_special_tokens=True)
    # Remove the prompt from the output
    answer = answer.replace(prompt, "").strip()
    return answer


In [16]:


# test with first sample question
context = "Open Value agreements are a simple, cost-effective way to acquire the latest Microsoft technology. These agreements are designed for small and medium organizations that want to license cloud services and on-premises software over a three-year period. Under Open Value agreements, organizations can elect to purchase perpetual licenses or subscribe to licenses and SA is included."
question = "What type of organizations is the Open Value agreements designed for and what licenses does it include?"
answer = ask_model(context, question)
print("Answer:", answer)

Answer: The Open Value agreements are designed for small and medium organizations and include the option to purchase perpetual licenses or subscribe to licenses with Software Assurance (SA) included.


In [21]:
context = "Mobility, one of the business units within the Communications segment, provides nationwide wireless service and equipment"
question = "What business segment of AT&T focuses on delivering nationwide wireless service and equipment?"
answer = ask_model(context, question)
print("Answer:", answer)

Answer: Mobility


In [18]:
context = "The Company allocates the transaction price to each performance obligation on a relative SSP basis. Judgment is required to determine the SSP for each distinct performance obligation. The Company determines SSP by considering its overall pricing objectives and market conditions. Significant pricing practices taken into consideration include the Company’s discounting practices, the size and volume of the Company’s transactions, the customer demographic, the geographic area where services are sold, price lists, the Company's go-to-market strategy, historical and current sales and contract prices."
question = "What is the basis for the Company to determine the Standalone Selling Price (SSP) for each distinct performance obligation in contracts with multiple performance obligations?"
answer = ask_model(context, question)
print("Answer:", answer)

Answer: The Company determines the Standalone Selling Price (SSP) for each distinct performance obligation on a relative SSP basis, considering its overall pricing objectives and market conditions, including its discounting practices, the size and volume of transactions, customer demographic, geographic area, price lists, and go-to-market strategy.


In [19]:

context = "As the rate implicit in the lease is rarely readily determinable, Delta Air Lines uses their incremental borrowing rate, which is based on the estimated interest rate for collateralized borrowing over a similar term of the lease at commencement date."
question = "What discount rate does Delta Air Lines use for lease payments when the rate implicit in the lease is not readily determinable?"
answer = ask_model(context, question)
print("Answer:", answer)

Answer: Delta Air Lines uses their incremental borrowing rate, which is based on the estimated interest rate for collateralized borrowing over a similar term of the lease at the commencement date.
